# Single CT scan to SPECTRE embeddings

This notebook shows a minimal, practical workflow to:

1. Load one CT scan and window it into crops with `spectre.load_and_window`.
2. Run the pretrained SPECTRE image model.
3. Do lightweight analysis on the resulting embeddings (norms, cosine similarity, PCA).

Preprocessing (HU windowing, RAS orientation, cropping, tiling) is handled by SPECTRE itself, so
this notebook only has to say *which* scan and *which* model.

> Reading `.nii`/`.nii.gz` needs the inference extra: `pip install "spectre-fm[inference]"`.
> If you would rather not write Python at all, `spectre embed scan.nii.gz -o out/` does the same
> thing from a terminal.

In [ ]:
# Only run this cell when working from a local clone without `pip install -e .`
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_src_paths = [cwd / "src", cwd.parent / "src"]
src_path = next((p for p in candidate_src_paths if p.exists()), None)

if src_path is None:
    print("Could not find a local src directory. Checked:")
    for p in candidate_src_paths:
        print(f"  - {p}")
elif str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    print(f"Added to sys.path: {src_path}")
else:
    print(f"src already on sys.path: {src_path}")

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from spectre import SpectreImageFeatureExtractor, list_presets, load_and_window, load_ct

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Available models:", list_presets())

## 1) Configure paths and model

Set the CT file path. The model weights download from Hugging Face the first time.

`SPACING = None` keeps the scan at its native voxel spacing, which is the SPECTRE default. Set it
to e.g. `(0.5, 0.5, 1.0)` to resample first.

In [ ]:
CT_PATH = Path(r"E:\Datasets\CT-RATE\dataset\valid\valid_1\valid_1_a\valid_1_a_1.nii.gz")
CROP_SIZE = (128, 128, 64)  # (H, W, D) - one crop, must match what the model was trained on
SPACING = None              # None = native spacing; or e.g. (0.5, 0.5, 1.0) to resample
MODEL_NAME = "spectre-large"

if not CT_PATH.exists():
    raise FileNotFoundError(
        f"CT file not found: {CT_PATH}\n"
        "Set CT_PATH to an existing .nii/.nii.gz file and rerun this cell."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print("Model:", MODEL_NAME)
print("Crop size:", CROP_SIZE)

## 2) Load and window the CT

`load_and_window` applies the pretraining pipeline in one call: load as RAS -> scale HU
[-1000, 1000] to [0, 1] -> optionally resample -> centre-crop to a whole multiple of the crop
size -> tile into non-overlapping crops.

It returns the crops `(N, C, H, W, D)`, the crop grid `(n_h, n_w, n_d)`, and the scan's metadata.

In [ ]:
crops, grid_size, meta = load_and_window(CT_PATH, crop_size=CROP_SIZE, spacing=SPACING)

print("Crops shape (N,C,H,W,D):", tuple(crops.shape))
print("Grid size (nH,nW,nD):", grid_size)
print("Voxel spacing (H,W,D) [mm]:", tuple(round(s, 4) for s in meta.spacing))
print("Original shape:", meta.original_shape, "| orientation:", meta.orientation)

# Reassemble the crops back into a volume, purely to visualise what the model actually sees.
n_h, n_w, n_d = grid_size
c_h, c_w, c_d = CROP_SIZE
volume = (
    crops.view(n_h, n_w, n_d, crops.shape[1], c_h, c_w, c_d)
    .permute(3, 0, 4, 1, 5, 2, 6)
    .reshape(crops.shape[1], n_h * c_h, n_w * c_w, n_d * c_d)
)
print("Windowed volume (C,H,W,D):", tuple(volume.shape))

spacing_h, spacing_w, spacing_d = meta.spacing

v = volume.squeeze(0).cpu().numpy()
cz, cy, cx = v.shape[2] // 2, v.shape[1] // 2, v.shape[0] // 2

# For imshow, aspect = row_spacing / column_spacing for true physical scaling.
axial_aspect = spacing_h / spacing_w
coronal_aspect = spacing_h / spacing_d
sagittal_aspect = spacing_w / spacing_d

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(v[:, :, cz], cmap="gray", aspect=axial_aspect)
axes[0].set_title("Axial (center)")
axes[1].imshow(v[:, cy, :], cmap="gray", aspect=coronal_aspect)
axes[1].set_title("Coronal (center)")
axes[2].imshow(v[cx, :, :], cmap="gray", aspect=sagittal_aspect)
axes[2].set_title("Sagittal (center)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3) Run SPECTRE and extract embeddings

`from_pretrained` downloads the weights and returns the model in eval mode.

Because the crops are already windowed, we pass them with their `grid_size`. To skip step 2
entirely you can hand the model a raw scan instead — `model(volume)` on a `(C, H, W, D)` tensor
in Hounsfield Units does the windowing internally and returns the same thing.

In [ ]:
model = SpectreImageFeatureExtractor.from_pretrained(MODEL_NAME, device=device)

with torch.no_grad():
    embeddings = model(crops, grid_size=grid_size)

embeddings_cpu = embeddings.detach().cpu()

print("Embeddings shape:", tuple(embeddings_cpu.shape), "= (1 CLS token + one per crop, features)")
print("Dtype:", embeddings_cpu.dtype)

# The first token summarises the whole scan; the rest map back onto the crop grid.
cls_token = embeddings_cpu[0]
patch_tokens = embeddings_cpu[1:].reshape(*grid_size, -1)
print("CLS token:", tuple(cls_token.shape), "| patch tokens:", tuple(patch_tokens.shape))

# Optional: save embeddings for downstream use
save_path = CT_PATH.parent / f"{CT_PATH.name.split('.')[0]}_spectre_embeddings.pt"
torch.save(
    {
        "ct_path": str(CT_PATH),
        "crop_size": CROP_SIZE,
        "grid_size": grid_size,
        "cls": cls_token,
        "patch_tokens": patch_tokens,
        "embeddings": embeddings_cpu,
    },
    save_path,
)
print("Saved embeddings to:", save_path)

## 4) Quick embedding analysis

This section provides a simple first look at the embedding geometry.

In [ ]:
# Tokens are already (T, F) for a single scan.
token_features = embeddings_cpu
if token_features.ndim != 2:
    raise RuntimeError(f"Unexpected embedding shape: {tuple(token_features.shape)}")

print("Token feature matrix:", tuple(token_features.shape))

# 1) Norm distribution
norms = token_features.norm(dim=-1).numpy()
print(f"Norm mean={norms.mean():.4f}, std={norms.std():.4f}, min={norms.min():.4f}, max={norms.max():.4f}")

plt.figure(figsize=(6, 4))
plt.hist(norms, bins=30)
plt.title("Embedding L2 norm distribution")
plt.xlabel("L2 norm")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# 2) Cosine similarity matrix between tokens
normed = F.normalize(token_features, dim=-1)
cosine = (normed @ normed.T).numpy()

plt.figure(figsize=(6, 5))
plt.imshow(cosine, cmap="viridis")
plt.colorbar(label="cosine similarity")
plt.title("Token-token cosine similarity")
plt.tight_layout()
plt.show()

# 3) PCA down to 2D for a quick look at the token geometry
centered = token_features - token_features.mean(dim=0, keepdim=True)
_, _, vh = torch.linalg.svd(centered, full_matrices=False)
projected = (centered @ vh[:2].T).numpy()

plt.figure(figsize=(6, 5))
plt.scatter(projected[:, 0], projected[:, 1], c=np.arange(len(projected)), cmap="plasma")
plt.colorbar(label="token index")
plt.title("Tokens projected onto their first 2 principal components")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()